# Evaluate Rewarded-Soups-Style LoRA Adapter Merges

This notebook tests fixed lambda-grid merging of two already trained GPT-2 LoRA adapters:

- a helpful adapter,
- a harmless adapter.

It loads the existing adapters, merges them across a fixed coefficient grid, generates responses, and saves the results for later analysis.

## 1. Check the GPU

In Colab, select **Runtime > Change runtime type > T4 GPU** before continuing. The code can run on CPU, but generation will be slower.

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. The notebook will use the CPU.")

## 2. Open or clone the repository

This cell always starts in `/content`. If the repository already exists, it updates it with `git pull`. Otherwise, it clones the repository. This avoids creating a nested path such as `/content/master-thesis/master-thesis`.

In [ ]:
%cd /content

from pathlib import Path
import subprocess

repo_path = Path("/content/master-thesis")

if repo_path.exists():
    print("Repository already exists. Pulling the latest changes...")
    subprocess.run(["git", "-C", str(repo_path), "pull"], check=True)
else:
    print("Cloning the repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/NZhang137/master-thesis.git"],
        check=True,
    )

%cd /content/master-thesis

## 3. Inspect the repository

You should see top-level entries such as `README.md`, `scripts/`, `src/`, `notebooks/`, and `results/`.

Inside `scripts/`, you should see:

- `train_hh_rlhf_adapters.py`
- `check_adapters.py`
- `evaluate_adapter_merges.py`

In [ ]:
!pwd
!ls

In [ ]:
!ls scripts
!ls src

## 4. Install dependencies

Install the libraries needed to load GPT-2 and PEFT LoRA adapters.

In [ ]:
# torchao is not needed for this GPT-2 + PEFT prototype.
!pip uninstall -y torchao
!pip install -q -U transformers datasets peft accelerate

The setup removes an old preinstalled `torchao` version to avoid compatibility errors with newer Transformers releases.

If `torchao` was already imported before running the installation cell, select **Runtime > Restart session**, then continue again from the repository setup and dependency installation cells.

## 5. Check whether the adapters already exist

If `adapters/` contains both `gpt2-helpful-adapter` and `gpt2-harmless-adapter`, continue to the adapter-file check. Otherwise, upload your local `adapters.zip` backup in the next step.

In [ ]:
!ls adapters || echo "No adapters folder found."

## 6. Upload `adapters.zip` if needed

`adapters.zip` is only a local backup of trained adapters. It must **not** be committed to GitHub.

Skip the upload cell if both adapter folders already exist. If they do not exist, run the upload cell and select `adapters.zip` from your computer.

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!if [ -f adapters.zip ]; then unzip -o adapters.zip; else echo "No adapters.zip found, skipping unzip."; fi

In [ ]:
!ls adapters

## 7. Check the adapter files

The checker confirms that both adapter folders contain the expected PEFT files:

- `adapter_config.json`
- `adapter_model.safetensors` (or the supported `.bin` alternative)

Continue only when both checks report `[OK]`.

In [ ]:
!python scripts/check_adapters.py

## 8. Run the fixed lambda-grid merge evaluation

The script evaluates these helpful/harmless coefficients:

- `[1.0, 0.0]`
- `[0.75, 0.25]`
- `[0.5, 0.5]`
- `[0.25, 0.75]`
- `[0.0, 1.0]`

For each coefficient pair, it generates responses to three fixed prompts and writes the small result table to `results/adapter_merge_generations.csv`.

In [ ]:
!python scripts/evaluate_adapter_merges.py

## 9. Inspect the results

The CSV should contain the helpful coefficient, harmless coefficient, prompt, and generated response. With five coefficient pairs and three prompts, it should contain 15 data rows.

In [ ]:
!ls results

In [ ]:
!head -n 10 results/adapter_merge_generations.csv

In [ ]:
import pandas as pd

df = pd.read_csv("results/adapter_merge_generations.csv")
df.head()

## 10. Git safety check

It is okay if `results/adapter_merge_generations.csv` appears as a new or modified file.

Do **not** commit:

- `adapters/`
- `adapters.zip`
- `.safetensors` or `.bin` files
- checkpoints or full model files

The repository ignore rules are intended to keep these generated artifacts out of Git.

In [ ]:
!git status

## What this notebook establishes

This notebook establishes that fixed Rewarded-Soups-style LoRA interpolation works technically for the two prototype adapters and produces generations for later pipeline stages.